In [0]:
#!/usr/bin/env python3
"""
Pushnami Data Engineering Exam — Data Generation Script

Generates 5 interconnected CSV datasets simulating a push notification
ad-tech platform. Contains intentional data quality issues that candidates
must discover during the exam.

Usage:
    python generate_data.py [--output-dir OUTPUT_DIR] [--seed SEED]
"""

import argparse
import csv
import os
import random
import uuid
from datetime import datetime, timedelta
import random

In [0]:
# ── Configuration ──────────────────────────────────────────────────────────

SEED = 42
OUTPUT_DIR = "output"

NUM_PARTNER_SITES = 50
NUM_SUBSCRIBERS = 100_000
NUM_CAMPAIGNS = 500
NUM_NOTIFICATIONS = 2_000_000
NUM_REVENUE_EVENTS = 200_000

# Date range for the dataset: 18 months of data
START_DATE = datetime(2023, 1, 1)
END_DATE = datetime(2024, 6, 30)

DEVICE_TYPES = ["desktop_chrome", "desktop_firefox", "desktop_edge",
                "android_chrome", "android_samsung", "ios_safari", "ios_chrome"]
CHANNELS = ["web_push", "mobile_push", "email", "in_app"]
CAMPAIGN_TYPES = ["promotional", "transactional", "re_engagement",
                  "onboarding", "breaking_news", "daily_digest"]
REGIONS = ["US-East", "US-West", "US-Central", "US-South", "EU-West",
           "EU-East", "APAC", "LATAM", "CA", "AU"]
ACQUISITION_SOURCES = ["organic", "paid_search", "social_media", "referral",
                       "partner_cross_promo", "email_campaign", "direct"]
VERTICALS = ["news", "ecommerce", "sports", "finance", "entertainment",
             "technology", "travel", "health", "education", "gaming"]

# Campaign ID format change cutover date (simulates system migration)
CAMPAIGN_FORMAT_CUTOVER = datetime(2023, 9, 15)

# Volume spike period (simulates a large partner onboarding)
SPIKE_START = datetime(2024, 2, 1)
SPIKE_END = datetime(2024, 2, 14)

# Timezone confusion window — events in this range may have Central timestamps
TZ_CONFUSION_START = datetime(2023, 6, 1)
TZ_CONFUSION_END = datetime(2023, 8, 31)

In [0]:
def parse_args():
    parser = argparse.ArgumentParser(description="Generate exam datasets")
    parser.add_argument("--output-dir", default=OUTPUT_DIR, help="Directory to write CSV files")
    parser.add_argument("--seed", type=int, default=SEED, help="Random seed for reproducibility")
    return parser.parse_args()

In [0]:
def random_date(start, end):
    delta = end - start
    random_seconds = random.randint(0, int(delta.total_seconds()))
    return start + timedelta(seconds=random_seconds)

In [0]:
def random_date_biased_recent(start, end):
    """Generate dates biased toward more recent — simulates organic growth."""
    delta = end - start
    # Square the random value to bias toward 1.0 (recent)
    factor = random.random() ** 0.7
    return start + timedelta(seconds=int(delta.total_seconds() * factor))

In [0]:
def generate_partner_sites(num_sites):
    sites = []
    tlds = [".com", ".net", ".org", ".io", ".co"]
    for i in range(num_sites):
        site_id = f"PS-{i+1:04d}"
        name_base = random.choice([
            "Daily", "Tech", "Sports", "News", "Shop", "Game", "Travel",
            "Health", "Finance", "Buzz", "Stream", "Quick", "Top", "Best",
            "Smart", "Fresh", "Prime", "Core", "Edge", "Peak"
        ])
        name_suffix = random.choice([
            "Wire", "Hub", "Zone", "Central", "Today", "Now", "World",
            "Watch", "Pulse", "Flash", "Post", "Feed", "Link", "Base"
        ])
        name = f"{name_base}{name_suffix}"
        domain = f"www.{name.lower()}{random.choice(tlds)}"

        onboard_date = random_date(START_DATE, END_DATE - timedelta(days=90))
        status = random.choices(["active", "paused", "churned"],
                                weights=[0.75, 0.15, 0.10])[0]
        vertical = random.choice(VERTICALS)
        monthly_uniques = random.randint(10000, 5000000)
        tier = ("enterprise" if monthly_uniques > 2000000
                else "professional" if monthly_uniques > 500000
                else "starter")
        revenue_share_pct = round(random.uniform(0.15, 0.45), 2)

        sites.append({
            "partner_site_id": site_id,
            "site_name": name,
            "domain": domain,
            "vertical": vertical,
            "onboard_date": onboard_date.strftime("%Y-%m-%d"),
            "status": status,
            "monthly_uniques": monthly_uniques,
            "tier": tier,
            "revenue_share_pct": revenue_share_pct,
            "primary_region": random.choice(REGIONS),
        })
    return sites

In [0]:

def generate_subscribers(num_subscribers, partner_sites):
    subscribers = []
    valid_site_ids = [s["partner_site_id"] for s in partner_sites]
    # Intentional: 3 phantom site IDs that don't exist in partner_sites
    phantom_sites = ["PS-0088", "PS-0091", "PS-0077"]

    for i in range(num_subscribers):
        sub_id = f"SUB-{i+1:07d}"
        # ~2% of subscribers assigned to phantom partner sites
        if random.random() < 0.02:
            site_id = random.choice(phantom_sites)
        else:
            site_id = random.choice(valid_site_ids)

        device = random.choice(DEVICE_TYPES)
        region = random.choice(REGIONS)
        source = random.choice(ACQUISITION_SOURCES)

        opt_in = random_date_biased_recent(START_DATE, END_DATE - timedelta(days=7))
        # ~1% null opt-in dates
        opt_in_str = opt_in.strftime("%Y-%m-%d %H:%M:%S") if random.random() > 0.01 else ""

        # Subscriber status
        days_since_optin = (END_DATE - opt_in).days
        if days_since_optin > 180:
            status = random.choices(["active", "inactive", "unsubscribed"],
                                    weights=[0.50, 0.30, 0.20])[0]
        else:
            status = random.choices(["active", "inactive", "unsubscribed"],
                                    weights=[0.75, 0.15, 0.10])[0]

        email_hash = uuid.uuid4().hex[:16]
        browser_lang = random.choices(
            ["en-US", "en-GB", "es-ES", "fr-FR", "de-DE", "pt-BR", "ja-JP"],
            weights=[0.55, 0.10, 0.10, 0.05, 0.05, 0.08, 0.07]
        )[0]

        subscribers.append({
            "subscriber_id": sub_id,
            "partner_site_id": site_id,
            "device_type": device,
            "region": region,
            "acquisition_source": source,
            "opt_in_date": opt_in_str,
            "status": status,
            "email_hash": email_hash,
            "browser_language": browser_lang,
        })

    # Inject ~2% duplicate subscribers with slight field variations
    num_dupes = int(num_subscribers * 0.02)
    for _ in range(num_dupes):
        original = random.choice(subscribers[:num_subscribers])
        dupe = dict(original)
        # Give the dupe a new subscriber_id but same email_hash
        dupe["subscriber_id"] = f"SUB-{num_subscribers + _ + 1:07d}"
        # Slight variations: case changes, whitespace
        variation = random.choice(["case", "whitespace", "region_case"])
        if variation == "case":
            dupe["device_type"] = dupe["device_type"].upper()
        elif variation == "whitespace":
            dupe["acquisition_source"] = " " + dupe["acquisition_source"]
        elif variation == "region_case":
            dupe["region"] = dupe["region"].lower()
        subscribers.append(dupe)

    return subscribers

In [0]:

def generate_campaigns(num_campaigns, partner_sites):
    campaigns = []
    valid_site_ids = [s["partner_site_id"] for s in partner_sites]

    for i in range(num_campaigns):
        created = random_date(START_DATE, END_DATE - timedelta(days=3))

        # Campaign ID format change mid-dataset
        if created < CAMPAIGN_FORMAT_CUTOVER:
            campaign_id = f"{10000 + i}"
        else:
            campaign_id = str(uuid.uuid4())

        camp_type = random.choice(CAMPAIGN_TYPES)
        channel = random.choice(CHANNELS)
        site_id = random.choice(valid_site_ids)

        # Target audience size
        audience_size = random.randint(500, 50000)

        # Schedule
        send_hour = random.choices(
            list(range(24)),
            weights=[1,1,1,1,1,2,3,5,8,10,10,8,7,6,7,8,9,10,8,6,4,3,2,1]
        )[0]
        send_date = created + timedelta(days=random.randint(0, 5))
        send_datetime = send_date.replace(hour=send_hour, minute=random.randint(0, 59))

        # Budget and bid
        budget = round(random.uniform(50, 5000), 2)
        bid_type = random.choice(["cpc", "cpm", "cpa"])
        bid_amount = round(random.uniform(0.01, 2.50), 4)

        status = random.choices(
            ["completed", "active", "scheduled", "paused", "cancelled"],
            weights=[0.55, 0.20, 0.10, 0.10, 0.05]
        )[0]

        campaigns.append({
            "campaign_id": campaign_id,
            "partner_site_id": site_id,
            "campaign_type": camp_type,
            "channel": channel,
            "audience_size": audience_size,
            "created_at": created.strftime("%Y-%m-%d %H:%M:%S"),
            "scheduled_send": send_datetime.strftime("%Y-%m-%d %H:%M:%S"),
            "budget": budget,
            "bid_type": bid_type,
            "bid_amount": bid_amount,
            "status": status,
        })

    return campaigns


In [0]:

def generate_notifications(num_notifications, subscribers, campaigns):
    """Generate notification event records with delivery funnel progression."""
    notifications = []
    campaign_list = list(campaigns)
    subscriber_list = list(subscribers)
    sub_count = len(subscriber_list)

    # Pre-compute subscriber indices for faster sampling
    for i in range(num_notifications):
        notif_id = f"NOTIF-{i+1:08d}"
        campaign = random.choice(campaign_list)
        subscriber = subscriber_list[random.randint(0, sub_count - 1)]

        # Determine send timestamp — base it on campaign scheduled_send
        base_time = datetime.strptime(campaign["scheduled_send"], "%Y-%m-%d %H:%M:%S")
        send_time = base_time + timedelta(
            minutes=random.randint(0, 120),
            seconds=random.randint(0, 59)
        )

        # Volume spike during the spike period
        if SPIKE_START <= send_time <= SPIKE_END:
            pass  # These naturally accumulate; we'll add extras below
        elif random.random() < 0.15:
            # Redistribute 15% of non-spike notifications into spike window
            if random.random() < 0.3:
                spike_offset = random.randint(0, int((SPIKE_END - SPIKE_START).total_seconds()))
                send_time = SPIKE_START + timedelta(seconds=spike_offset)

        # Timezone inconsistency: ~5% of events during confusion window
        # are stored with US/Central offset (-6h) without indication
        tz_note = ""
        if TZ_CONFUSION_START <= send_time <= TZ_CONFUSION_END:
            if random.random() < 0.05:
                send_time = send_time - timedelta(hours=6)
                tz_note = "CST"

        # Delivery funnel
        delivered = random.random() < 0.92
        clicked = delivered and random.random() < 0.045
        converted = clicked and random.random() < 0.12

        # Event timestamps
        delivered_at = ""
        clicked_at = ""
        converted_at = ""

        if delivered:
            delivered_at = (send_time + timedelta(seconds=random.randint(1, 30))).strftime("%Y-%m-%d %H:%M:%S")
        if clicked:
            clicked_at = (send_time + timedelta(seconds=random.randint(31, 3600))).strftime("%Y-%m-%d %H:%M:%S")
        if converted:
            converted_at = (send_time + timedelta(seconds=random.randint(61, 7200))).strftime("%Y-%m-%d %H:%M:%S")

        notifications.append({
            "notification_id": notif_id,
            "campaign_id": campaign["campaign_id"],
            "subscriber_id": subscriber["subscriber_id"],
            "partner_site_id": campaign["partner_site_id"],
            "channel": campaign["channel"],
            "sent_at": send_time.strftime("%Y-%m-%d %H:%M:%S"),
            "delivered_at": delivered_at,
            "clicked_at": clicked_at,
            "converted_at": converted_at,
        })

        if i % 500000 == 0 and i > 0:
            print(f"  ...generated {i:,} / {num_notifications:,} notifications")

    return notifications


In [0]:

def generate_revenue_events(num_events, notifications):
    """Generate revenue events tied to converted notifications."""
    revenue_events = []

    # Collect converted notifications
    converted = [n for n in notifications if n["converted_at"]]
    if not converted:
        print("WARNING: No converted notifications found")
        return revenue_events

    # We'll generate more revenue events than conversions by allowing
    # multiple revenue events per conversion (upsells, etc.)
    orphan_count = 0
    for i in range(num_events):
        rev_id = f"REV-{i+1:07d}"

        # ~3% orphaned revenue events with non-existent notification_ids
        if random.random() < 0.03:
            notif_id = f"NOTIF-{90000000 + i:08d}"
            campaign_id = "UNKNOWN"
            subscriber_id = "UNKNOWN"
            partner_site_id = "UNKNOWN"
            base_time = random_date(START_DATE, END_DATE)
            orphan_count += 1
        else:
            notif = random.choice(converted)
            notif_id = notif["notification_id"]
            campaign_id = notif["campaign_id"]
            subscriber_id = notif["subscriber_id"]
            partner_site_id = notif["partner_site_id"]
            base_time = datetime.strptime(notif["converted_at"], "%Y-%m-%d %H:%M:%S")

        event_time = base_time + timedelta(seconds=random.randint(0, 300))

        # Revenue amount
        revenue_type = random.choices(
            ["purchase", "subscription", "ad_click", "lead"],
            weights=[0.35, 0.25, 0.30, 0.10]
        )[0]

        if revenue_type == "purchase":
            amount = round(random.uniform(5.0, 250.0), 2)
        elif revenue_type == "subscription":
            amount = round(random.choice([4.99, 9.99, 14.99, 29.99, 49.99]), 2)
        elif revenue_type == "ad_click":
            amount = round(random.uniform(0.05, 3.00), 2)
        else:
            amount = round(random.uniform(1.00, 50.00), 2)

        # ~0.5% negative revenue (refunds) — intentionally NOT labeled
        if random.random() < 0.005:
            amount = -abs(amount)

        currency = random.choices(
            ["USD", "EUR", "GBP", "CAD", "AUD"],
            weights=[0.70, 0.12, 0.08, 0.05, 0.05]
        )[0]

        revenue_events.append({
            "revenue_id": rev_id,
            "notification_id": notif_id,
            "campaign_id": campaign_id,
            "subscriber_id": subscriber_id,
            "partner_site_id": partner_site_id,
            "event_timestamp": event_time.strftime("%Y-%m-%d %H:%M:%S"),
            "revenue_type": revenue_type,
            "amount": amount,
            "currency": currency,
        })

        if i % 50000 == 0 and i > 0:
            print(f"  ...generated {i:,} / {num_events:,} revenue events")

    print(f"  (includes {orphan_count:,} orphaned revenue events)")
    return revenue_events

In [0]:

def write_csv(filepath, data, fieldnames):
    with open(filepath, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(data)
    print(f"  Wrote {len(data):,} rows to {filepath}")

In [0]:
seed = 42
random.seed(seed)

#output_dir = args.output_dir
#output_dir = "/Workspace/Users/reveriano.christopher@gmail.com/DatabrickP1/"
#output_dir = "/Workspace/Users/reveriano.christopher@gmail.com/Data/"
output_dir = "/Workspace/Users/reveriano.christopher@gmail.com/DatabrickP1/Data/"
os.makedirs(output_dir, exist_ok=True)


print("=" * 60)
print("Pushnami Data Engineering Exam — Data Generator")
print("=" * 60)

    # ── Partner Sites ──
print("\n[1/5] Generating partner sites...")
partner_sites = generate_partner_sites(NUM_PARTNER_SITES)
write_csv(
    os.path.join(output_dir, "partner_sites.csv"),
    partner_sites,
    ["partner_site_id", "site_name", "domain", "vertical", "onboard_date",
    "status", "monthly_uniques", "tier", "revenue_share_pct", "primary_region"],
)

In [0]:
print("\n[2/5] Generating subscribers...")
subscribers = generate_subscribers(NUM_SUBSCRIBERS, partner_sites)
write_csv(
    os.path.join(output_dir, "subscribers.csv"),
    subscribers,
    ["subscriber_id", "partner_site_id", "device_type", "region",
    "acquisition_source", "opt_in_date", "status", "email_hash",
    "browser_language"],
)

In [0]:
print("\n[3/5] Generating campaigns...")
campaigns = generate_campaigns(NUM_CAMPAIGNS, partner_sites)
write_csv(
    os.path.join(output_dir, "campaigns.csv"),
    campaigns,
    ["campaign_id", "partner_site_id", "campaign_type", "channel",
    "audience_size", "created_at", "scheduled_send", "budget",
    "bid_type", "bid_amount", "status"],
)

In [0]:
print("\n[4/5] Generating notifications (this may take a minute)...")
notifications = generate_notifications(NUM_NOTIFICATIONS, subscribers, campaigns)
write_csv(
    os.path.join(output_dir, "notifications.csv"),
    notifications,
    ["notification_id", "campaign_id", "subscriber_id", "partner_site_id",
    "channel", "sent_at", "delivered_at", "clicked_at", "converted_at"],
)

In [0]:
print("\n[5/5] Generating revenue events...")
revenue_events = generate_revenue_events(NUM_REVENUE_EVENTS, notifications)
write_csv(
    os.path.join(output_dir, "revenue_events.csv"),
    revenue_events,
    ["revenue_id", "notification_id", "campaign_id", "subscriber_id",
    "partner_site_id", "event_timestamp", "revenue_type", "amount",
    "currency"],
)


In [0]:
print("\n" + "=" * 60)
print("Generation complete! Summary:")
print(f"  Partner Sites:    {len(partner_sites):>10,}")
print(f"  Subscribers:      {len(subscribers):>10,}")
print(f"  Campaigns:        {len(campaigns):>10,}")
print(f"  Notifications:    {len(notifications):>10,}")
print(f"  Revenue Events:   {len(revenue_events):>10,}")
print(f"\nOutput directory: {os.path.abspath(output_dir)}")
print("=" * 60)